# Sea Surface Temperature comparison between MOM6-NEP10k and observational data (OISST) in CCIEA domain
Author: Brooke Hawkins

For details and code to query the MOM6-NEP10k data with Earthmover, see notebook `sst-anomaly.ipynb`. This notebook compares the results.

## Set up

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

config = {
    "mom6_path": "mom6/mom6_monthly_sst.nc",
    "oisst_path": "oisst/TS_monthly.nc",
    "mom6_var": "tos",
    "oisst_var": "sst",
    "spatial_bounds": {"lat": [32, 48], "lon": [-128, -116.8]},
    "lon_offset": 360,
    "time_bounds": {"start": None, "end": None},
}


## Load data

In [ ]:
# load datasets
sst_mom6 = xr.open_dataset(config["mom6_path"])
sst_oisst = xr.open_dataset(config["oisst_path"])


I confirmed that the MOM6 subset here has duplicates for the last six months of data, January to June of 2025. Each time step is duplicated once, and the values are identical. I need to fix this in the `sst-anomaly.ipynb` script before the data is written to netCDF.

In [ ]:
# remove duplicates from MOM6
sst_mom6 = sst_mom6.drop_duplicates(dim="time", keep="first")


## Define spatial and temporal extents

The default time bounds are the start and end dates for MOM6 hindcast. In this case, that's useful because the MOM6-NEP10k hindcast has a shorter available time frame than OISST.

In [ ]:
# define spatial extent for California Current
lat_bounds = config["spatial_bounds"]["lat"]
lon_bounds = config["spatial_bounds"]["lon"]

# define temporal extent for comparison, default to MOM6
time_bgn = (
    config["time_bounds"]["start"]
    if config["time_bounds"]["start"] is not None
    else sst_mom6.time.to_index().min()
)
time_end = (
    config["time_bounds"]["end"]
    if config["time_bounds"]["end"] is not None
    else sst_mom6.time.to_index().max()
)

print(f"Date range is {time_bgn} to {time_end}")


In [ ]:
# subset both datasets (add longitude offset for 0-360 coordinate scale)
sst_mom6_subset = sst_mom6.sel(
    lat_vec=slice(lat_bounds[0], lat_bounds[1]),
    lon_vec=slice(
        lon_bounds[0] + config["lon_offset"], lon_bounds[1] + config["lon_offset"]
    ),
    time=slice(time_bgn, time_end),
)
sst_oisst_subset = sst_oisst.sel(
    lat_vec=slice(lat_bounds[0], lat_bounds[1]),
    lon_vec=slice(
        lon_bounds[0] + config["lon_offset"], lon_bounds[1] + config["lon_offset"]
    ),
    time=slice(time_bgn, time_end),
)


In [ ]:
# check grid sizes
print("MOM6 dimensions: ", sst_mom6_subset.sizes)
print("OISST dimensions: ", sst_oisst_subset.sizes)


## Check grid resolutions and interpolate

In [ ]:
def check_grid_resolution(dataset):
    """
    Analyzes the spatial resolution of a dataset's grid to determine if it is
    regular (uniform spacing), nearly regular (close to uniform spacing, some
    rounding error), or irregular.

    Args:
        dataset: An object (e.g., xarray Dataset) containing 'lat_vec' and 'lon_vec'
                 coordinate arrays that can be converted to pandas Series.
    """
    # check step size between adjacent latitude points to check for uniform spacing
    lat_series = dataset.lat_vec.to_series()
    lat_series_diff = lat_series.diff()
    lat_diff_min = lat_series_diff.min(skipna=True)
    lat_diff_max = lat_series_diff.max(skipna=True)

    # check step size between adjacent longitude points to check for uniform spacing
    lon_series = dataset.lon_vec.to_series()
    lon_series_diff = lon_series.diff()
    lon_diff_min = lon_series_diff.min(skipna=True)
    lon_diff_max = lon_series_diff.max(skipna=True)

    # classify grid resolution based on variance in step sizes
    if lat_diff_min == lat_diff_max and lon_diff_min == lon_diff_max:
        print(
            f"Perfectly regular grid resolution of {lat_diff_min} latitudinal degrees by {lon_diff_min} longitudinal degrees."
        )
    elif np.isclose(lat_diff_min, lat_diff_max) and np.isclose(
        lon_diff_min, lon_diff_max
    ):
        print(
            f"Nearly regular grid resolution of {lat_diff_min} latitudinal degrees by {lon_diff_min} longitudinal degrees."
        )
    else:
        print(
            f"Irregular grid resolution of {lat_diff_min} to {lat_diff_max} latitudinal degrees by {lon_diff_min} to {lon_diff_max} longitudinal degrees."
        )

In [ ]:
check_grid_resolution(sst_oisst_subset)
check_grid_resolution(sst_mom6_subset)

Since MOM6 has a higher resolution, I will interpolate the MOM6 data to the OISST grid. I use bilinear interpolation using xarray's `interp_like` function. In other applications, other kinds of interpolation may be more appropriate.

In [ ]:
# interpolate MOM6 grid onto OISST grid
sst_mom6_interp = sst_mom6_subset.interp_like(sst_oisst_subset)

# check interpolated MOM6 grid size and resolution
print("MOM6 dimensions: ", sst_mom6_interp.sizes)
check_grid_resolution(sst_mom6_interp)


Calculate weights based on latitude to area-weight statistics. The area of each 1/2 degree by 1/2 degree grid cell varies with latitude, since the distance that one degree of longitude represents shrinks when moving from the equator to the poles. If comparison statistics are not area weighted, then smaller cells further north (which are generally cooler waters) will bias comparisons.

In [ ]:
# calculate latitude-based area weights
weights = np.cos(np.deg2rad(sst_oisst_subset.lat_vec))


## Calculate comparisons

Calculate means, bias, correlation, and root mean squared error (RMSE). Calculate three kinds of summaries:
* global: across all space and time (single number)
* temporal: across space (use to make time series)
* spatial: across time (use to make map)

### Means

In [ ]:
# extract SST data
mom6_data = sst_mom6_interp[config["mom6_var"]]
oisst_data = sst_oisst_subset[config["oisst_var"]]

# calculate averages
mom6_weighted = mom6_data.weighted(weights)
oisst_weighted = oisst_data.weighted(weights)

# make sure to area weight calculation across spatial dimensions
mom6_mean_global = mom6_weighted.mean().item()
oisst_mean_global = oisst_weighted.mean().item()
mom6_mean_temporal = mom6_weighted.mean(dim=["lat_vec", "lon_vec"])
oisst_mean_temporal = oisst_weighted.mean(dim=["lat_vec", "lon_vec"])
# no need to area weight calculation across time dimension
mom6_spatial = mom6_data.mean(dim="time")
oisst_spatial = oisst_data.mean(dim="time")


### Bias

In [ ]:
# calculate bias
error = mom6_data - oisst_data
error_weighted = error.weighted(weights)
# make sure to area weight calculation across spatial dimensions
bias_global = error_weighted.mean().item()
bias_temporal = error_weighted.mean(dim=["lat_vec", "lon_vec"])
# no need to area weight calculation across time dimension
bias_spatial = error.mean(dim="time")


### Correlation

In [ ]:
# calculate correlation
# make sure to area weight calculation across spatial dimensions
correlation_global = xr.corr(oisst_data, mom6_data, weights=weights).item()
correlation_temporal = xr.corr(
    oisst_data, mom6_data, dim=["lat_vec", "lon_vec"], weights=weights
)
# no need to area weight calculation across time dimension
correlation_spatial = xr.corr(oisst_data, mom6_data, dim="time")


### RMSE

In [ ]:
# calculate RMSE
squared_error = error**2
squared_error_weighted = squared_error.weighted(weights)
# make sure to area weight calculation across spatial dimensions
rmse_global = np.sqrt((squared_error_weighted).mean()).item()
rmse_temporal = np.sqrt((squared_error_weighted).mean(dim=["lat_vec", "lon_vec"]))
# no need to area weight calculation across time dimension
rmse_spatial = np.sqrt((squared_error).mean(dim="time"))


## Visualize comparisons

In [ ]:
# print table summary of global bias, RMSE, and correlation
summary = pd.DataFrame(
    {
        "Statistic": ["Bias", "Correlation", "RMSE"],
        "Value": [bias_global, correlation_global, rmse_global],
    }
)
print(summary)

In the California Current, sea surface temperature from MOM6-NEP10k runs slightly warmer (0.14°C) than observational data in OISST, is highly correlated (R = 0.97), and generally falls within 1 degree of OISST (RMSE = 0.70°C).

Compared to the model domain comparison ([Drenkard et al. 2025](https://doi.org/10.5194/gmd-18-5245-2025), Figure 2a-c), the CCIEA domain runs warmer rather than cooler (bias = -0.16°C), is similarly correlated (R = 1.00), and falls within 1/2 degree of OISST (RMSE = 0.28).

In [ ]:
# visualize time series with average temperatures and bias

# create a figure with 2 rows and 1 column, sharing the x-axis
fig, (ax1, ax2) = plt.subplots(
    nrows=2, ncols=1, figsize=(10, 6), layout="constrained", sharex=True
)

# subplot a) average temperatures for MOM6 and OISST
mom6_mean_temporal.plot(ax=ax1, label="MOM6", color="tab:blue")
oisst_mean_temporal.plot(ax=ax1, label="OISST", color="tab:orange")
ax1.legend()
ax1.set_ylabel("SST (°C)")
ax1.set_xlabel("")

# subplot b) bias
bias_temporal.plot(ax=ax2, color="black")
ax2.set_ylabel("Bias (°C)")
ax2.set_xlabel("Year")

# add grid
ax1.grid(True, linestyle="--", alpha=0.6)
ax2.grid(True, linestyle="--", alpha=0.6)

# display plot
plt.show()

In the California Current, MOM6-NEP10k captures seasonal variability in sea surface temperature (time series are well aligned), and is slightly more variable (with higher highs, and lower lows) than OISST.

In [ ]:
# visualize maps with average temperatures and bias

# create a figure with 1 row and 3 columns
fig, axes = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(12, 6),
    layout="constrained",
    subplot_kw={"projection": ccrs.PlateCarree()},
)

# determine shared min and max limits for the MOM6 and OISST temperature plots
vmin = min(mom6_spatial.min().item(), oisst_spatial.min().item())
vmax = max(mom6_spatial.max().item(), oisst_spatial.max().item())

# colormaps
cmap_sst = "turbo"  # 'jet' or 'nipy_spectral' also work for blue-to-red
cmap_bias = "RdBu_r"  # Blue to White to Red diverging colormap

# subplot a) SST (MOM6)
im1 = mom6_spatial.plot(
    ax=axes[0],
    cmap=cmap_sst,
    vmin=vmin,
    vmax=vmax,
    add_colorbar=False,
    transform=ccrs.PlateCarree(),
)
axes[0].set_title("a) MOM6 SST")

# subplot b) SST (OISSST)
im2 = oisst_spatial.plot(
    ax=axes[1],
    cmap=cmap_sst,
    vmin=vmin,
    vmax=vmax,
    add_colorbar=False,
    transform=ccrs.PlateCarree(),
)
axes[1].set_title("b) OISST SST")

# subplot c) difference (bias)
im3 = bias_spatial.plot(
    ax=axes[2],
    cmap=cmap_bias,
    center=0,  # lock white to 0
    add_colorbar=False,
    transform=ccrs.PlateCarree(),
)
axes[2].set_title("c) Difference (MOM6 - OISST)")

# coastline
land1 = cfeature.NaturalEarthFeature(
    "physical", "land", "50m", edgecolor="slategray", facecolor="lightgray"
)

# axis formatting
for i, ax in enumerate(axes):
    # add coastline to all subplots
    ax.add_feature(land1, linewidth=0.5, zorder=40)

    # set map extent for all subplots
    ax.set_extent(
        [
            config["spatial_bounds"]["lon"][0],
            config["spatial_bounds"]["lon"][1],
            config["spatial_bounds"]["lat"][0],
            config["spatial_bounds"]["lat"][1],
        ],
        crs=ccrs.PlateCarree(),
    )

    # remove axis labels to all subplots
    ax.set_xlabel("")
    ax.set_ylabel("")

    # set tick locations
    lon_ticks = np.arange(
        config["spatial_bounds"]["lon"][0],
        config["spatial_bounds"]["lon"][1],
        2,
    )
    lat_ticks = np.arange(
        config["spatial_bounds"]["lat"][0], config["spatial_bounds"]["lat"][1], 2
    )
    ax.set_xticks(lon_ticks, crs=ccrs.PlateCarree())
    ax.set_yticks(lat_ticks, crs=ccrs.PlateCarree())

    # add longitude labels to all subplots
    ax.xaxis.set_major_formatter(LongitudeFormatter(zero_direction_label=True))

    if i == 0:
        # add latitude labels to first subplot
        ax.yaxis.set_major_formatter(LatitudeFormatter())
    else:
        # remove y-axis label for subplots b and c
        ax.set_yticklabels([])

# custom horizontal colorbars
# shared colorbar for subplots a and b
fig.colorbar(
    im1,
    ax=[axes[0], axes[1]],  # center between panels a and b
    location="bottom",
    label="SST (°C)",
    shrink=0.4,
)
# individual colorbar for subplot c
fig.colorbar(im3, ax=axes[2], location="bottom", label="Difference (°C)", shrink=0.8)

# display plot
plt.show()

In the California Current, MOM6-NEP10k captures spatial variability in sea surface temperature. Spatial patterns are similar, with cooler waters off the Pacific Northwest coast, and progressively warmer surface temperatures to the south and east. There are two areas of warmer bias ~0.5°C concentrated near Monterey Bay and south of the Channel Islands.